# 06 — Staffing Optimisation and Decision Layer

Forecasting is only useful if it changes decisions.

This notebook converts demand forecasts into recommended staffing levels for clinicians, nurses and front-desk staff.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)


In [ ]:
from clinic_forecast.data import generate_synthetic_healthcare_data
from clinic_forecast.models.baseline import seasonal_naive_forecast
from clinic_forecast.staffing import StaffingRules, recommend_staffing, staffing_gap

data_path = PROJECT_ROOT / "data" / "raw" / "clinic_usage.csv"
metadata_path = PROJECT_ROOT / "data" / "raw" / "clinic_metadata.csv"

if data_path.exists() and metadata_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
    metadata = pd.read_csv(metadata_path)
else:
    usage, metadata, _ = generate_synthetic_healthcare_data()
    usage["date"] = pd.to_datetime(usage["date"])

cutoff = usage["date"].max() - pd.Timedelta(days=28)
train = usage[usage["date"] <= cutoff]
future = usage[usage["date"] > cutoff]
forecast = seasonal_naive_forecast(train=train, future=future)
forecast.head()


## Staffing rules

The rules below are assumptions. In a real healthcare network, these would be calibrated with operational leaders and historical service-level targets.


In [ ]:
rules = StaffingRules(
    visits_per_clinician_day=18,
    visits_per_nurse_day=24,
    visits_per_frontdesk_day=35,
    minimum_clinicians=1,
    minimum_nurses=1,
    minimum_frontdesk=1,
    buffer_ratio=0.12,
)

staffing = recommend_staffing(forecast, rules=rules)
staffing.head()


In [ ]:
staffing_with_gap = staffing_gap(staffing, metadata)
staffing_with_gap[[
    "clinic_id", "date", "forecast", "recommended_clinicians", "base_clinicians", "clinician_gap",
    "recommended_nurses", "base_nurses", "nurse_gap",
    "recommended_frontdesk", "base_frontdesk", "frontdesk_gap",
]].head()


## Network staffing load

The staffing recommendation can be aggregated across the full network or by region. This is useful for workforce planning and temporary staffing allocation.


In [ ]:
daily_staffing = staffing_with_gap.groupby("date", as_index=False).agg(
    clinicians=("recommended_clinicians", "sum"),
    nurses=("recommended_nurses", "sum"),
    frontdesk=("recommended_frontdesk", "sum"),
    clinician_gap=("clinician_gap", "sum"),
    nurse_gap=("nurse_gap", "sum"),
    frontdesk_gap=("frontdesk_gap", "sum"),
)

fig, ax = plt.subplots()
ax.plot(daily_staffing["date"], daily_staffing["clinicians"], label="clinicians")
ax.plot(daily_staffing["date"], daily_staffing["nurses"], label="nurses")
ax.plot(daily_staffing["date"], daily_staffing["frontdesk"], label="frontdesk")
ax.set_title("Recommended network staffing")
ax.set_xlabel("Date")
ax.set_ylabel("Staff count")
ax.legend()
plt.show()

daily_staffing.head()


## Operational report

The final output is a clinic-day staffing table. This can feed an internal dashboard, scheduling tool or API.


In [ ]:
output_dir = PROJECT_ROOT / "reports" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
staffing_with_gap.to_csv(output_dir / "staffing_recommendations.csv", index=False)
print(f"Saved staffing recommendations to {output_dir / 'staffing_recommendations.csv'}")
